# 构造full-text，调用LLM处理

## 步骤一.导入所需库

In [2]:
import pandas as pd
import numpy as np
import os
import torch
from transformers import BertTokenizerFast, BertModel
from tqdm import tqdm
import pickle
import warnings

# 忽略 pandas 中关于 .iloc[] 赋值的常见警告
pd.options.mode.chained_assignment = None  # default='warn'
warnings.filterwarnings('ignore', category=UserWarning, module='transformers')

## 步骤 2：定义常量和路径

In [3]:
# --- 路径定义 ---
RAW_DATA_FILE = os.path.join('..', 'data', 'raw', 'california_raw_data.csv')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
OUTPUT_FILE = os.path.join(PROCESSED_DIR, 'california_nlp_features.pkl')

# --- 模型定义 ---
# 您可以选择 "bert-base-uncased" (768维) 或 "bert-large-uncased" (1024维)
# "bert-base-uncased" 对大多数任务来说速度和性能都很好
MODEL_NAME = 'bert-base-uncased'

# --- 硬件设置 ---
# 检查是否有可用的 GPU，否则使用 CPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"将使用以下设备进行处理: {DEVICE.upper()}")

将使用以下设备进行处理: CPU


## 步骤 3：加载原始数据

In [4]:
try:
    df_raw = pd.read_csv(RAW_DATA_FILE, low_memory=False)
    print(f"成功加载原始数据: {RAW_DATA_FILE}")
    print(f"原始数据形状: {df_raw.shape}")
    print(df_raw.head())
except FileNotFoundError:
    print(f"错误: 原始数据文件未找到. 请先运行 '01_data_acquisition.ipynb'")
    # 如果在 .py 脚本中，这里应该使用 raise 退出

成功加载原始数据: ..\data\raw\california_raw_data.csv
原始数据形状: (11383, 161)
  reportingrailroadcode                reportingrailroadname  year  \
0                  BNSF                 BNSF Railway Company  1997   
1                  BNSF                 BNSF Railway Company  1999   
2                  BNSF                 BNSF Railway Company  2015   
3                  SJVR  San Joaquin Valley Railroad Company  2011   
4                  BNSF                 BNSF Railway Company  2011   

  accidentnumber                                                url  \
0      SC1097111  {'url': 'https://safetydata.fra.dot.gov/Office...   
1      SC0699123  {'url': 'https://safetydata.fra.dot.gov/Office...   
2      CA0815200  {'url': 'https://safetydata.fra.dot.gov/Office...   
3       ID110340  {'url': 'https://safetydata.fra.dot.gov/Office...   
4      CA0911203  {'url': 'https://safetydata.fra.dot.gov/Office...   

   accidentyear  accidentmonth maintenancerailroadcode  \
0            97            

## 步骤 4：数据清理（移除零方差列）

In [5]:
# 查找所有列中唯一值的数量
unique_counts = df_raw.nunique()

# 筛选出唯一值数量为 1 的列（即零方差列）
cols_to_drop = unique_counts[unique_counts == 1].index.tolist()

if cols_to_drop:
    # 从 DataFrame 中删除这些列
    df = df_raw.drop(columns=cols_to_drop)
    print(f"\n移除了 {len(cols_to_drop)} 个零方差列:")
    print(cols_to_drop)
else:
    df = df_raw
    print("\n未发现零方差列。")

print(f"清理后数据形状: {df.shape}")


移除了 4 个零方差列:
['statecode', 'stateabbr', 'statename', 'district']
清理后数据形状: (11383, 157)


## 步骤 5：构造“全文表示”

In [6]:
# 步骤 5.1: 将所有 NaN 值替换为字符串 'NA'
# LLM 会将 'NA' 作为一个词元来学习其含义
df_filled = df.fillna('NA')

# 步骤 5.2: 遍历每一行并创建长字符串
full_text_list = []
for _, row in tqdm(df_filled.iterrows(), total=df.shape[0], desc="[1/3] 构造全文表示"):
    text_parts = []
    for col_name in df_filled.columns:
        # 格式: "[列名] [值], "
        text_parts.append(f"{col_name} {row[col_name]},")
    
    # 将所有部分用空格连接成一个长字符串
    full_text_list.append(" ".join(text_parts))

print(f"\n已成功为 {len(full_text_list)} 条记录创建了全文表示。")

# 打印第一个示例以供验证
if full_text_list:
    print("\n--- 全文表示示例 (第1条记录): ---")
    print(full_text_list[0][:500] + "...") # 仅打印前500个字符

[1/3] 构造全文表示: 100%|██████████| 11383/11383 [00:02<00:00, 4709.35it/s]


已成功为 11383 条记录创建了全文表示。

--- 全文表示示例 (第1条记录): ---
reportingrailroadcode BNSF, reportingrailroadname BNSF Railway Company, year 1997, accidentnumber SC1097111, url {'url': 'https://safetydata.fra.dot.gov/Officeofsafety/Publicsite/FORM54/F54Report.aspx?RepType=SQL&txtf54key=BNSFSC109711119971021'}, accidentyear 97, accidentmonth 10, maintenancerailroadcode BNSF, maintenancerailroadname BNSF Railway Company, maintenanceaccidentnumber SC1097111, maintenanceaccidentyear 97, maintenanceaccidentmonth 10, day 21, date 1997-10-21T00:00:00.000, time 1:05...


## 步骤 6：加载 LLM (BERT) 模型和分词器

In [7]:
print(f"\n正在从 Hugging Face 加载 '{MODEL_NAME}'...")
# 1. 加载分词器
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)

# 2. 加载模型
model = BertModel.from_pretrained(MODEL_NAME)

# 3. 将模型发送到 GPU (如果可用)
model.to(DEVICE)
model.eval()  # 将模型设置为评估模式（这对于推理很重要）
print("模型和分词器加载完毕。")


正在从 Hugging Face 加载 'bert-base-uncased'...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

d:\MyProjects\ResearchProjects\railway_risk_analysis\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\14704\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

模型和分词器加载完毕。


## 步骤 7：生成特征嵌入（分批处理）

In [8]:
BATCH_SIZE = 32  # 您可以根据您 GPU 显存大小调整此值
all_embeddings = []

# 使用 tqdm 显示批处理进度
for i in tqdm(range(0, len(full_text_list), BATCH_SIZE), desc="[2/3] 生成特征嵌入"):
    
    # 1. 获取当前批次的文本
    batch_texts = full_text_list[i : i + BATCH_SIZE]
    
    # 2. 批量分词 (Tokenize)
    inputs = tokenizer(
        batch_texts, 
        padding=True,        # 填充到批次中的最大长度
        truncation=True,     # 截断超过512个token的文本
        max_length=512,      # BERT 的最大长度
        return_tensors='pt'  # 返回 PyTorch 张量
    )
    
    # 3. 将输入数据发送到 GPU
    inputs = {key: val.to(DEVICE) for key, val in inputs.items()}
    
    # 4. 模型推理 (Inference)
    with torch.no_grad():  # 在推理时关闭梯度计算
        outputs = model(**inputs)
        
    # 5. 提取 [CLS] 向量 (pooler_output) 并移回 CPU
    batch_embeddings = outputs.pooler_output.cpu().numpy()
    
    # 6. 收集结果
    all_embeddings.append(batch_embeddings)

# 7. 将所有批次的向量堆叠成一个大的 NumPy 矩阵
X_nlp = np.vstack(all_embeddings)

print("\n特征嵌入生成完毕。")
print(f"最终特征矩阵 (X_nlp) 形状: {X_nlp.shape}")

[2/3] 生成特征嵌入: 100%|██████████| 356/356 [57:45<00:00,  9.73s/it]  


特征嵌入生成完毕。
最终特征矩阵 (X_nlp) 形状: (11383, 768)


## 步骤 8：保存处理后的特征
将这个高维的NumPy矩阵 X_nlp 保存到 data/processed 文件夹中。使用 pickle 来保存，因为它可以高效地序列化NumPy数组。

In [9]:
# 确保 'data/processed' 目录存在
os.makedirs(PROCESSED_DIR, exist_ok=True)

# 使用 pickle 保存 NumPy 矩阵
try:
    with open(OUTPUT_FILE, 'wb') as f:
        pickle.dump(X_nlp, f)
    print(f"\n[3/3] 成功！处理后的特征已保存到: {OUTPUT_FILE}")
except IOError as e:
    print(f"错误: 保存文件失败. {e}")


[3/3] 成功！处理后的特征已保存到: ..\data\processed\california_nlp_features.pkl
